In [ ]:
# original brfss extraction, by col+manual

pct_medical_cost <- read.csv("out/medical_cost.csv") %>%
  select(PHR, group, pct, low, high, answer)
pct_medical_cost <- pct_medical_cost %>%
  filter(group == "Total" & answer == "YES")
pct_medical_cost <- pct_medical_cost[, !names(pct_medical_cost) %in% c("group", "answer", "low", "high")]
pct_medical_cost <- pct_medical_cost %>% rename(medical_cost_pct = pct)
print(pct_medical_cost)

past_flu_shot <- read.csv("out/flu_shot.csv") %>%
  select(PHR, group, pct, low, high, answer)
past_flu_shot <- past_flu_shot %>%
  filter(group == "Total" & answer == "YES")
past_flu_shot <- past_flu_shot[, !names(past_flu_shot) %in% c("group", "answer", "low", "high")]
past_flu_shot <- past_flu_shot %>% rename(flu_shot_pct = pct)
print(past_flu_shot)

brfss_data <- pct_medical_cost %>%
  inner_join(past_flu_shot, by = "PHR")
print(brfss_data)
write.csv(brfss_data, file = "data/brfss_data.csv", row.names = FALSE)

pneumonia_shot <- read.csv("out/pneumonia_shot.csv") %>%
  select(PHR, group, pct, low, high, answer)
pneumonia_shot <- pneumonia_shot %>%
  filter(group == "Total" & answer == "YES")
pneumonia_shot <- pneumonia_shot[, !names(pneumonia_shot) %in% c("group", "answer", "low", "high")]
pneumonia_shot <- pneumonia_shot %>% rename(pneumonia_shot_pct = pct)
print(pneumonia_shot)

brfss_data <- read.csv("data/brfss_data.csv")
print(brfss_data)
brfss_data <- brfss_data %>%
  inner_join(pneumonia_shot, by = "PHR")
print(brfss_data)
write.csv(brfss_data, file = "data/brfss_data.csv", row.names = FALSE)

routine_check <- read.csv("out/routine_check.csv") %>%
  select(PHR, group, pct, low, high, answer)
routine_check <- routine_check %>%
  filter(group == "Total" & answer == "Within the past year")
routine_check <- routine_check[, !names(routine_check) %in% c("group", "answer", "low", "high")]
routine_check <- routine_check %>% rename(routine_check_pct = pct)
print(routine_check)


In [ ]:
# Second try on brfss extraction, only having D10 response
from pathlib import Path
import pandas as pd

SHEET = "Routine Checkup"

def read_medical_cost_block(xlsx_path: Path) -> pd.DataFrame:
    # Excel row 9 is header -> header=8; keep rows 10–37 -> nrows=28
    df = pd.read_excel(
        xlsx_path,
        sheet_name=SHEET,
        header=8,
        nrows=28,
        engine="openpyxl"
    )

    cols = list(df.columns)
    df = df.rename(columns={
        cols[0]: "group",
        cols[1]: "category",
        cols[2]: "sample_size",
        cols[3]: "yes_pct",
        cols[4]: "yes_ci",
        cols[5]: "no_pct",
        cols[6]: "no_ci",
    })

    df["group"] = df["group"].ffill()
    return df[["group","category","sample_size","yes_pct","yes_ci","no_pct","no_ci"]]

def ci_to_low_high(ci_series: pd.Series) -> pd.DataFrame:
    tokens = ci_series.astype(str).str.extract(
        r"^\(\s*([0-9.]+|\.)\s*-\s*([0-9.]+|\.)\s*\)$"
    )
    low = pd.to_numeric(tokens[0].replace(".", pd.NA), errors="coerce")
    high = pd.to_numeric(tokens[1].replace(".", pd.NA), errors="coerce")
    return pd.DataFrame({"ci_low": low, "ci_high": high})

def to_long_yes_no(df: pd.DataFrame) -> pd.DataFrame:
    yes = pd.DataFrame({
        "group": df["group"],
        "category": df["category"],
        "sample_size": df["sample_size"],
        "response": "YES",
        "pct": pd.to_numeric(df["yes_pct"], errors="coerce"),
    }).join(ci_to_low_high(df["yes_ci"]))

    no = pd.DataFrame({
        "group": df["group"],
        "category": df["category"],
        "sample_size": df["sample_size"],
        "response": "NO",
        "pct": pd.to_numeric(df["no_pct"], errors="coerce"),
    }).join(ci_to_low_high(df["no_ci"]))

    return pd.concat([yes, no], ignore_index=True)

def process_one_file(xlsx_path: Path) -> pd.DataFrame:
    base = read_medical_cost_block(xlsx_path)
    long = to_long_yes_no(base)
    long.insert(0, "source_file", xlsx_path.name)
    return long

def process_11_at_once(input_dir: str, output_csv: str, pattern: str="*.xlsx") -> pd.DataFrame:
    in_dir = Path(input_dir)
    files = sorted([p for p in in_dir.glob(pattern) if p.suffix.lower()==".xlsx" and not p.name.startswith("~$")])
    if len(files) == 0:
        raise FileNotFoundError(f"No .xlsx files matched {pattern} in {in_dir}")

    all_long = pd.concat([process_one_file(p) for p in files], ignore_index=True)

    out_path = Path(output_csv)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    all_long.to_csv(out_path, index=False)
    return all_long

# Example:
df_all = process_11_at_once(
    input_dir=r"data/brfss",
    output_csv=r"out/intermediate.csv",
    pattern="*.xlsx"
)
print(df_all.shape)